# 03 - UNet Reconstruction for Sparse-View CT

This notebook implements the learned post-processing stage of the sparse-view CT pipeline. It loads the fixed degraded sinograms, converts them to image-domain FBP inputs in memory, trains one UNet for each sparse-view setting, and saves trained weights, metrics, reconstructions, and figures.


## Environment and Project Paths

The notebook is configured for execution in Google Colab with project data stored on Google Drive. The path definitions identify the processed dataset, the model weight directory, and the output folders for figures, metrics, and reconstructions. ASTRA is installed in the runtime because the CT projection and FBP operations require it.


In [ ]:
!pip install astra-toolbox

The implementation uses PyTorch for the neural network and optimization loop, NumPy for angular discretization, Matplotlib for visual inspection, and IPPy for CT operators, FBP reconstruction, device selection, and image-quality metrics.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

import csv
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

PROJECT_ROOT = Path("/content/drive/MyDrive/LM_INFORMATICA/COMPUTATIONAL_IMAGING")
IPPY_DIR = PROJECT_ROOT / "IPPy"
PROCESSED_DIR = PROJECT_ROOT / "processed"
WEIGHTS_DIR = PROJECT_ROOT / "weights" / "unet"
FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"
METRICS_DIR = PROJECT_ROOT / "outputs" / "metrics"
RECONSTRUCTIONS_DIR = PROJECT_ROOT / "outputs" / "reconstructions" / "unet"

for directory in [WEIGHTS_DIR, FIGURES_DIR, METRICS_DIR, RECONSTRUCTIONS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

sys.path.append(str(PROJECT_ROOT))

from IPPy import operators, solvers, utilities
from IPPy.utilities import normalize
from IPPy.utilities import metrics as ippy_metrics

SEED = 42
IMAGE_SIZE = 256
ANGLE_COUNTS = (180, 90, 60, 45)
DETECTOR_SIZE = 256
GEOMETRY = "parallel"
NUM_EPOCHS = 20
LEARNING_RATE = 1e-3
BASE_CHANNELS = 32
BATCH_SIZE = 8


torch.manual_seed(SEED)
np.random.seed(SEED)
device = utilities.get_device()

print("Device:", device)
print("Project root:", PROJECT_ROOT)
print("Processed data:", PROCESSED_DIR)
print("UNet weights:", WEIGHTS_DIR)


## Processed Data Contract

The processed dataset contains one PyTorch file per Mayo patient. Each file stores clean images, degraded sinograms for the four sparse-view configurations, original source paths, and metadata. The UNet stage reads these files directly and does not regenerate measurement data.


In [ ]:
manifest_path = PROCESSED_DIR / "manifest.json"
with manifest_path.open("r", encoding="utf-8") as file:
    manifest = json.load(file)


def collect_processed_patients(split_name: str) -> list[Path]:
    return [PROCESSED_DIR / record["path"] for record in manifest["splits"][split_name]["patients"]]


processed_patients = {
    "train": collect_processed_patients("train"),
    "test": collect_processed_patients("test"),
}

for split_name, patient_paths in processed_patients.items():
    num_images = manifest["splits"][split_name]["num_images"]
    print(f"{split_name}: {num_images} images, {len(patient_paths)} patient files")

example_payload = torch.load(processed_patients["train"][0], map_location="cpu")
print("Example clean shape:", tuple(example_payload["clean"].shape))
for angle_key in [str(n) for n in ANGLE_COUNTS]:
    print(f"Example sinogram {angle_key}:", tuple(example_payload["sinograms"][angle_key].shape))


## In-Memory FBP Domain Transform

The neural network operates in the image domain, while the measured data are sinograms. A filtered backprojection transform is therefore applied to each noisy sinogram mini-batch before the UNet forward pass. The resulting FBP images contain sparse-view artifacts and act as the network input, while the corresponding clean images provide the supervised target.


In [ ]:
projectors = {
    str(n_views): operators.CTProjector(
        img_shape=(IMAGE_SIZE, IMAGE_SIZE),
        angles=np.linspace(0.0, np.pi, n_views),
        det_size=DETECTOR_SIZE,
        geometry=GEOMETRY,
    )
    for n_views in ANGLE_COUNTS
}

fbp_solvers = {angle_key: solvers.FBP(projector) for angle_key, projector in projectors.items()}


def normalize_batch_per_image(batch: torch.Tensor) -> torch.Tensor:
    normalized = []
    for image in batch:
        normalized.append(normalize(image.unsqueeze(0)).clamp(0.0, 1.0).to(torch.float32))
    return torch.cat(normalized, dim=0)


@torch.no_grad()
def compute_fbp_proxy(sinogram: torch.Tensor, angle_key: str) -> torch.Tensor:
    fbp, _ = fbp_solvers[angle_key](sinogram, x_true=None, starting_point=None)
    return normalize_batch_per_image(fbp.detach().cpu())


sample_fbp = compute_fbp_proxy(example_payload["sinograms"]["60"][:BATCH_SIZE], "60")
print("Sample FBP proxy shape:", tuple(sample_fbp.shape))



## Dataset and DataLoader

Each dataset item corresponds to one processed patient file. The loader returns clean slices and the requested noisy sinograms. During training and evaluation, each patient tensor is split into mini-batches; FBP is computed only for the active mini-batch and kept in memory. This avoids an additional proxy dataset on Drive while keeping memory usage controlled.


In [ ]:
class PatientSinogramDataset(Dataset):
    def __init__(self, patient_paths: list[Path], angle_key: str):
        self.patient_paths = list(patient_paths)
        self.angle_key = angle_key

    def __len__(self):
        return len(self.patient_paths)

    def __getitem__(self, idx):
        patient_path = self.patient_paths[idx]
        payload = torch.load(patient_path, map_location="cpu")
        return {
            "sinogram": payload["sinograms"][self.angle_key].detach().cpu().to(torch.float32),
            "clean": payload["clean"].detach().cpu().to(torch.float32),
            "source_paths": payload["source_paths"],
            "metadata": payload["metadata"],
            "patient_path": str(patient_path),
        }


def make_loader(patient_paths: list[Path], angle_key: str, shuffle: bool) -> DataLoader:
    dataset = PatientSinogramDataset(patient_paths, angle_key)
    return DataLoader(dataset, batch_size=None, shuffle=shuffle, num_workers=0)


def iter_slice_batches(patient_batch: dict, batch_size: int = BATCH_SIZE):
    sinogram = patient_batch["sinogram"]
    clean = patient_batch["clean"]
    for start in range(0, clean.shape[0], batch_size):
        end = start + batch_size
        yield sinogram[start:end], clean[start:end]


example_loader = make_loader(processed_patients["train"], "60", shuffle=True)
example_patient = next(iter(example_loader))
example_sinogram_batch, example_clean_batch = next(iter_slice_batches(example_patient))
example_fbp_batch = compute_fbp_proxy(example_sinogram_batch, "60")
print("Patient sinogram tensor:", tuple(example_patient["sinogram"].shape))
print("Slice batch sinogram:", tuple(example_sinogram_batch.shape))
print("Slice batch FBP input:", tuple(example_fbp_batch.shape))
print("Slice batch clean target:", tuple(example_clean_batch.shape))
print("Example metadata:", example_patient["metadata"])


## UNet Architecture

The model follows a standard encoder-decoder UNet design. The encoder progressively extracts features at lower spatial resolutions, while the decoder restores the image resolution. Skip connections concatenate encoder and decoder features, helping preserve anatomical edges and local structures while removing sparse-view artifacts.


In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1),
            nn.ReLU(),
        )

    def forward(self, x):
        return self.block(x)


class DownBlock(nn.Module):
    def __init__(self, in_ch, out_ch, block_cls=DoubleConv):
        super().__init__()
        self.pool = nn.MaxPool2d(2)
        self.block = block_cls(in_ch, out_ch)

    def forward(self, x):
        return self.block(self.pool(x))


class UpBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch, block_cls=DoubleConv):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, out_ch, kernel_size=2, stride=2)
        self.block = block_cls(out_ch + skip_ch, out_ch)

    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[-2:] != skip.shape[-2:]:
            x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        x = torch.cat([skip, x], dim=1)
        return self.block(x)


class UNet(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, base_ch=32):
        super().__init__()
        self.enc1 = DoubleConv(in_ch, base_ch)
        self.enc2 = DownBlock(base_ch, 2 * base_ch)
        self.enc3 = DownBlock(2 * base_ch, 4 * base_ch)
        self.bottleneck = DownBlock(4 * base_ch, 8 * base_ch)
        self.dec3 = UpBlock(8 * base_ch, 4 * base_ch, 4 * base_ch)
        self.dec2 = UpBlock(4 * base_ch, 2 * base_ch, 2 * base_ch)
        self.dec1 = UpBlock(2 * base_ch, base_ch, base_ch)
        self.out_conv = nn.Conv2d(base_ch, out_ch, kernel_size=1)

    def forward(self, x):
        s1 = self.enc1(x)
        s2 = self.enc2(s1)
        s3 = self.enc3(s2)
        h = self.bottleneck(s3)
        h = self.dec3(h, s3)
        h = self.dec2(h, s2)
        h = self.dec1(h, s1)
        return self.out_conv(h)


test_model = UNet(in_ch=1, out_ch=1, base_ch=BASE_CHANNELS)
num_params = sum(param.numel() for param in test_model.parameters())
print(f"UNet parameters: {num_params / 1e6:.2f}M")

## Training Loop

A separate UNet is trained for each sparse-view setting. For each patient file, noisy sinograms are transformed into FBP mini-batches in memory and used as network inputs. The clean slices are the supervised targets. Since the pipeline uses only the provided train and test splits, the best model is selected by the lowest training loss and saved under `weights/unet`.


In [ ]:
def train_unet_for_angle(angle_key: str) -> list[float]:
    train_loader = make_loader(processed_patients["train"], angle_key, shuffle=True)
    model = UNet(in_ch=1, out_ch=1, base_ch=BASE_CHANNELS).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    loss_fn = nn.MSELoss()
    history = []
    best_loss = float("inf")
    best_path = WEIGHTS_DIR / f"unet_{angle_key}.pt"
    final_path = WEIGHTS_DIR / f"unet_{angle_key}_final.pt"

    for epoch in range(NUM_EPOCHS):
        model.train()
        running_loss = 0.0
        num_samples = 0
        progress = tqdm(train_loader, desc=f"UNet {angle_key} epoch {epoch + 1}/{NUM_EPOCHS}")

        for patient_batch in progress:
            for sinogram_batch, clean_batch in iter_slice_batches(patient_batch):
                fbp_input = compute_fbp_proxy(sinogram_batch, angle_key).to(device)
                clean_target = clean_batch.to(device)

                optimizer.zero_grad(set_to_none=True)
                prediction = model(fbp_input)
                loss = loss_fn(prediction, clean_target)
                loss.backward()
                optimizer.step()

                batch_size = clean_target.shape[0]
                running_loss += loss.item() * batch_size
                num_samples += batch_size
                progress.set_postfix(loss=running_loss / num_samples)

        epoch_loss = running_loss / num_samples
        history.append(epoch_loss)

        if epoch_loss < best_loss:
            best_loss = epoch_loss
            torch.save(
                {
                    "angle_key": angle_key,
                    "model_state_dict": model.state_dict(),
                    "epoch": epoch + 1,
                    "best_loss": best_loss,
                    "history": history,
                    "config": {
                        "base_channels": BASE_CHANNELS,
                        "learning_rate": LEARNING_RATE,
                        "num_epochs": NUM_EPOCHS,
                        "batch_size": BATCH_SIZE,
                    },
                },
                best_path,
            )

    torch.save(
        {
            "angle_key": angle_key,
            "model_state_dict": model.state_dict(),
            "epoch": NUM_EPOCHS,
            "final_loss": history[-1],
            "history": history,
        },
        final_path,
    )
    print(f"Saved best weights for {angle_key} views:", best_path)

    del model
    del optimizer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return history


training_histories = {}
for angle_key in [str(n) for n in ANGLE_COUNTS]:
    training_histories[angle_key] = train_unet_for_angle(angle_key)


## Training Curves

The training curves summarize the mean squared error across epochs for each sparse-view configuration. They provide a compact view of optimization behavior and make it possible to compare training difficulty across different numbers of projection angles.


In [ ]:
training_curve_path = FIGURES_DIR / "unet_training_curves.png"

fig, ax = plt.subplots(figsize=(7, 4), constrained_layout=True)
for angle_key, history in training_histories.items():
    ax.plot(range(1, len(history) + 1), history, marker="o", label=f"{angle_key} views")

ax.set_xlabel("Epoch")
ax.set_ylabel("Training MSE")
ax.set_title("UNet training curves")
ax.grid(True, alpha=0.3)
ax.legend()
fig.savefig(training_curve_path, dpi=150)
plt.show()
plt.close(fig)

print("Saved training curves:", training_curve_path)

## Evaluation Metrics and Reconstruction Export

Each trained model is evaluated on the matching test-angle sinograms. FBP inputs are computed in memory, UNet outputs are compared against the clean ground truth with PSNR and SSIM, and the final UNet reconstructions are saved for later comparison. FBP proxy images are not persisted as separate files.


In [ ]:
def load_best_model(angle_key: str) -> UNet:
    checkpoint = torch.load(WEIGHTS_DIR / f"unet_{angle_key}.pt", map_location=device)
    model = UNet(in_ch=1, out_ch=1, base_ch=BASE_CHANNELS).to(device)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()
    return model


def compute_batch_metrics(prediction: torch.Tensor, clean: torch.Tensor) -> dict[str, float]:
    prediction = torch.clamp(prediction.detach().cpu(), 0.0, 1.0)
    clean = torch.clamp(clean.detach().cpu(), 0.0, 1.0)
    return {
        "psnr": float(ippy_metrics.PSNR(prediction, clean)),
        "ssim": float(ippy_metrics.SSIM(prediction, clean)),
    }


@torch.no_grad()
def evaluate_angle(angle_key: str) -> tuple[dict[str, float], dict]:
    model = load_best_model(angle_key)
    test_loader = make_loader(processed_patients["test"], angle_key, shuffle=False)
    totals = {
        "fbp_psnr": 0.0,
        "fbp_ssim": 0.0,
        "unet_psnr": 0.0,
        "unet_ssim": 0.0,
    }
    num_samples = 0
    first_example = None

    output_dir = RECONSTRUCTIONS_DIR / angle_key
    output_dir.mkdir(parents=True, exist_ok=True)

    for patient_batch in tqdm(test_loader, desc=f"Evaluate {angle_key}"):
        metadata = patient_batch["metadata"]
        patient_id = metadata["patient_id"]
        patient_predictions = []
        patient_clean = []

        for sinogram_batch, clean_batch in iter_slice_batches(patient_batch):
            fbp_input = compute_fbp_proxy(sinogram_batch, angle_key).to(device)
            clean_target = clean_batch.to(device)
            prediction = torch.clamp(model(fbp_input), 0.0, 1.0)

            fbp_metrics = compute_batch_metrics(fbp_input, clean_target)
            unet_metrics = compute_batch_metrics(prediction, clean_target)
            batch_size = clean_target.shape[0]

            totals["fbp_psnr"] += fbp_metrics["psnr"] * batch_size
            totals["fbp_ssim"] += fbp_metrics["ssim"] * batch_size
            totals["unet_psnr"] += unet_metrics["psnr"] * batch_size
            totals["unet_ssim"] += unet_metrics["ssim"] * batch_size
            num_samples += batch_size

            patient_predictions.append(prediction.detach().cpu())
            patient_clean.append(clean_target.detach().cpu())

            if first_example is None:
                first_example = {
                    "fbp": fbp_input[:1].detach().cpu(),
                    "unet": prediction[:1].detach().cpu(),
                    "clean": clean_target[:1].detach().cpu(),
                }

        reconstruction_path = output_dir / f"{patient_id}.pt"
        torch.save(
            {
                "unet": torch.cat(patient_predictions, dim=0),
                "clean": torch.cat(patient_clean, dim=0),
                "source_paths": patient_batch["source_paths"],
                "metadata": {**metadata, "angle_key": angle_key},
            },
            reconstruction_path,
        )

    averaged = {key: value / num_samples for key, value in totals.items()}
    averaged["num_samples"] = num_samples

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return averaged, first_example


evaluation_results = {}
comparison_examples = {}
for angle_key in [str(n) for n in ANGLE_COUNTS]:
    metrics_for_angle, example = evaluate_angle(angle_key)
    evaluation_results[angle_key] = metrics_for_angle
    comparison_examples[angle_key] = example
    print(angle_key, metrics_for_angle)



## Metrics Table

The metrics table reports both the FBP input quality and the UNet output quality for each sparse-view setting. This CSV file is used by the final comparison notebook to aggregate quantitative results across reconstruction methods.


In [ ]:
metrics_path = METRICS_DIR / "unet_metrics.csv"
with metrics_path.open("w", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(["angle", "num_samples", "fbp_psnr", "fbp_ssim", "unet_psnr", "unet_ssim"])
    for angle_key, values in evaluation_results.items():
        writer.writerow(
            [
                angle_key,
                values["num_samples"],
                values["fbp_psnr"],
                values["fbp_ssim"],
                values["unet_psnr"],
                values["unet_ssim"],
            ]
        )

print("Saved UNet metrics:", metrics_path)

## Visual Comparison with Detail Crop

The comparison figures show the FBP input, the UNet reconstruction, and the ground truth side by side. A central crop is included to make local artifact removal and edge preservation easier to assess visually.


In [ ]:
def save_angle_comparison(angle_key: str, example: dict) -> None:
    crop = (slice(96, 160), slice(96, 160))
    images = {
        "Input FBP": torch.clamp(example["fbp"][0, 0], 0.0, 1.0),
        "Output UNet": torch.clamp(example["unet"][0, 0], 0.0, 1.0),
        "Ground Truth": torch.clamp(example["clean"][0, 0], 0.0, 1.0),
    }

    fig, axes = plt.subplots(2, 3, figsize=(10, 7), constrained_layout=True)
    for col, (title, image) in enumerate(images.items()):
        axes[0, col].imshow(image, cmap="gray", vmin=0.0, vmax=1.0)
        axes[0, col].set_title(title)
        axes[0, col].axis("off")

        axes[1, col].imshow(image[crop], cmap="gray", vmin=0.0, vmax=1.0)
        axes[1, col].set_title(f"{title} crop")
        axes[1, col].axis("off")

    fig.suptitle(f"UNet learned post-processing - {angle_key} views")
    output_path = FIGURES_DIR / f"unet_comparison_{angle_key}.png"
    fig.savefig(output_path, dpi=150)
    plt.show()
    plt.close(fig)
    print("Saved comparison:", output_path)


for angle_key, example in comparison_examples.items():
    save_angle_comparison(angle_key, example)